In [2]:
from pathlib import Path

print(Path.cwd())
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

DATA_DIR.mkdir(exist_ok=True)

print("Dataset location:", DATA_DIR)

C:\Users\user\deep-learning-traffic-signs\notebooks
Dataset location: C:\Users\user\deep-learning-traffic-signs\data


In [3]:
from torchvision import transforms 
basic_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [4]:
from torchvision.datasets import GTSRB

train_dataset = GTSRB(
    root=str(DATA_DIR),
    split="train",
    transform=basic_transform,
    download=True
)

test_dataset = GTSRB(
    root=str(DATA_DIR),
    split="test",
    transform=basic_transform,
    download=True
)

In [5]:
from torch.utils.data import DataLoader

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [7]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
 
)

In [8]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([32, 3, 32, 32])
Labels shape: torch.Size([32])


In [9]:
batch=next(iter(train_loader))
batch

[tensor([[[[0.5725, 0.5647, 0.5725,  ..., 0.6039, 0.6078, 0.6196],
           [0.5647, 0.5686, 0.5725,  ..., 0.6118, 0.6118, 0.6235],
           [0.5686, 0.5725, 0.5647,  ..., 0.6078, 0.6196, 0.6275],
           ...,
           [0.5412, 0.5412, 0.5333,  ..., 0.5490, 0.5569, 0.5529],
           [0.5412, 0.5412, 0.5373,  ..., 0.5412, 0.5451, 0.5451],
           [0.5333, 0.5373, 0.5333,  ..., 0.5451, 0.5373, 0.5412]],
 
          [[0.6941, 0.6902, 0.6863,  ..., 0.7176, 0.7255, 0.7255],
           [0.6902, 0.6980, 0.6941,  ..., 0.7216, 0.7216, 0.7216],
           [0.6902, 0.6941, 0.6941,  ..., 0.7176, 0.7255, 0.7255],
           ...,
           [0.6353, 0.6431, 0.6353,  ..., 0.6392, 0.6431, 0.6471],
           [0.6353, 0.6353, 0.6353,  ..., 0.6392, 0.6392, 0.6392],
           [0.6392, 0.6314, 0.6314,  ..., 0.6353, 0.6392, 0.6353]],
 
          [[0.9137, 0.9098, 0.9020,  ..., 0.9176, 0.9255, 0.9294],
           [0.8941, 0.9059, 0.9020,  ..., 0.9216, 0.9216, 0.9255],
           [0.9059, 0.90

In [10]:
import torch

In [11]:
device=torch.device("cuda"if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [12]:
import torch
import torch.nn as nn
from torch.optim import Adam
import  torchvision.transforms.v2 as transforms
import torchvision.transforms.functional as F 
import matplotlib.pyplot as plt


In [13]:
class MyConvBlock( nn.Module ):
    def __init__(self,in_ch,out_ch,dropout_p):
        kernel_size=3
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,kernel_size,stride=1,padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.MaxPool2d(2,stride=2)
        )
    def forward(self,x):
        return self.model(x)
        

In [14]:
flattened_img_size=128*4*4
N_CLASSES=43
IMG_CHS=3
IMG_WIDH=32
IMG_LENGHT=32
base_model= nn.Sequential(
    MyConvBlock(IMG_CHS,32,0), #(32,16,16)
    MyConvBlock(32,64,0.2),#(64,8,8)
    MyConvBlock(64,128,0),#(128,4,4)
    nn.Flatten(),
    nn.Linear(flattened_img_size,512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512,N_CLASSES)
)
    

In [15]:
loss_function=nn.CrossEntropyLoss()
optimizer=Adam(base_model.parameters())

In [16]:
train_N=len(train_loader.dataset)
test_N=len(test_loader.dataset)

In [17]:

import torch
torch._dynamo.config.suppress_errors = True
model=torch.compile(base_model.to(device))
model

OptimizedModule(
  (_orig_mod): Sequential(
    (0): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0, inplace=False)
        (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(128, eps=1e

In [18]:
def get_batch_accuracy(output,y,N):
    pred=output.argmax(dim=1,keepdim=True)
    correct=pred.eq(y.view_as(pred)).sum().item()
    return correct/N


In [19]:
def train():
    loss=0
    accuracy=0
    model.train()
    for x,y in train_loader:
        output=model(x)
        optimizer.zero_grad()
        batch_loss=loss_function(output,y)
        batch_loss.backward()
        optimizer.step()
        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output,y,train_N)
    print('Valid-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))

In [20]:
def validate():
    loss=0
    accuracy=0
    model.eval()
    with torch.no_grad():
        for x,y in test_loader:
            output=model(x)
            loss+=loss_function(output,y).item()
            accuracy += get_batch_accuracy(output,y,test_N)
        print('Valid-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))

In [21]:
epochs=20
for epoch in range (epochs):
    print ('epoch:{}'.format(epoch))   
    train()
    validate()

epoch:0


W0921 19:38:54.285000 22792 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] WON'T CONVERT inner C:\Users\user\deep-learning-traffic-signs\.venv\Lib\site-packages\torch\_dynamo\external_utils.py line 67 
W0921 19:38:54.285000 22792 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] due to: 
W0921 19:38:54.285000 22792 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] Traceback (most recent call last):
W0921 19:38:54.285000 22792 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]   File "C:\Users\user\deep-learning-traffic-signs\.venv\Lib\site-packages\torch\_dynamo\convert_frame.py", line 2333, in __call__
W0921 19:38:54.285000 22792 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]     result = self._inner_convert(
W0921 19:38:54.285000 22792 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]         frame, cache_entry, hooks, frame_state, skip=skip + 1
W0921 19:38:54.285000 22792 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]     )
W0921 19:38:54.2

Valid-Loss:943.7859 Accuracy 0.6589
Valid-Loss:207.7994 Accuracy 0.8358
epoch:1
Valid-Loss:130.2583 Accuracy 0.9512
Valid-Loss:130.9997 Accuracy 0.9067
epoch:2
Valid-Loss:67.8160 Accuracy 0.9755
Valid-Loss:96.3276 Accuracy 0.9282
epoch:3
Valid-Loss:56.0635 Accuracy 0.9778
Valid-Loss:79.3097 Accuracy 0.9452
epoch:4
Valid-Loss:42.5703 Accuracy 0.9839
Valid-Loss:107.4209 Accuracy 0.9314
epoch:5
Valid-Loss:34.4051 Accuracy 0.9864
Valid-Loss:83.4161 Accuracy 0.9432
epoch:6
Valid-Loss:32.5354 Accuracy 0.9878
Valid-Loss:75.3056 Accuracy 0.9584
epoch:7
Valid-Loss:27.9428 Accuracy 0.9906
Valid-Loss:80.8940 Accuracy 0.9527
epoch:8
Valid-Loss:25.3893 Accuracy 0.9907
Valid-Loss:58.4154 Accuracy 0.9635
epoch:9
Valid-Loss:24.9787 Accuracy 0.9914
Valid-Loss:76.7293 Accuracy 0.9511
epoch:10
Valid-Loss:17.1665 Accuracy 0.9941
Valid-Loss:62.8973 Accuracy 0.9624
epoch:11
Valid-Loss:22.1329 Accuracy 0.9924
Valid-Loss:75.6205 Accuracy 0.9529
epoch:12
Valid-Loss:17.5131 Accuracy 0.9936
Valid-Loss:72.0710 Ac